# Cyclone Mocha — Sentinel Asia Value-Added Products (May 2023)

Cyclone Mocha made landfall near the Bangladesh/Myanmar border on **14 May 2023**.  
This notebook visualises the three flood-extent Value-Added Products (VAPs) published by
[Sentinel Asia](https://sentinel-asia.org/EO/2023/article20230514BD.html) and hosted as
ArcGIS tiled map services:

| Layer | Sensor | Date | Producer |
|---|---|---|---|
| JAXA Flood Proxy Map (automatic) | ALOS-2 SAR | 15 May 2023 | JAXA |
| Detected Water — Chittagong Province | ALOS-2 SAR | 15 May 2023 | AIT |
| Flooded Areas — Bangladesh | Sentinel-1 SAR | 18 May 2023 | MBRSC |

The services are tiled PNG map services (XYZ/WMTS) served from  
`https://tiles.arcgis.com/tiles/Ce7cHX9rBcQKrxER/arcgis/rest/services/{name}/MapServer/tile/{z}/{y}/{x}`

We load them as HoloViews `WMTS` tile sources and overlay them on a satellite basemap.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import holoviews as hv
import geoviews as gv
import geoviews.tile_sources as gvts
from holoviews import opts

hv.extension('bokeh')
gv.extension('bokeh')

# Study area — Bangladesh coast / Chittagong region
# Web Mercator bounds covering all three VAP extents
XLIM = (9_900_000, 10_500_000)   # ~lon 88.9 – 94.3
YLIM = (2_050_000, 2_650_000)    # ~lat 18.2 – 23.2

TILE_BASE = (
    "https://tiles.arcgis.com/tiles/Ce7cHX9rBcQKrxER"
    "/arcgis/rest/services/{name}/MapServer/tile/{Z}/{Y}/{X}"
)

SERVICES = {
    "JAXA — ALOS-2 (15 May)": "JAXA_by_ALOS2_20230515",
    "AIT — ALOS-2 (15 May)": "AIT_by_ALOS2_20230515",
    "MBRSC — Sentinel-1 (18 May)": "MBRSC_by_Sentinel1_20230518",
}

print("Packages loaded")

## 2. Individual VAP layers

In [ ]:
def vap_tile(service_name):
    url = TILE_BASE.format(name=service_name, Z="{Z}", Y="{Y}", X="{X}")
    return gv.WMTS(url)

basemap = gvts.EsriImagery

shared_opts = opts.WMTS(
    width=500, height=450,
    xlim=XLIM, ylim=YLIM,
)

plots = []
for title, svc in SERVICES.items():
    layer = basemap * vap_tile(svc)
    layer = layer.opts(
        shared_opts,
        opts.Overlay(title=title),
    )
    plots.append(layer)

hv.Layout(plots).cols(2)

## 3. Combined overlay — all three VAPs

Stack all three flood layers on a single satellite basemap to compare spatial coverage.

In [ ]:
combined = basemap
for svc in SERVICES.values():
    combined = combined * vap_tile(svc)

combined.opts(
    opts.WMTS(width=750, height=650, xlim=XLIM, ylim=YLIM),
    opts.Overlay(title="All VAPs — Cyclone Mocha flood extent (May 2023)"),
)

## 4. Interactive selector — choose layer with a widget

In [ ]:
import panel as pn
pn.extension()

layer_select = pn.widgets.Select(
    name="VAP layer",
    options=list(SERVICES.keys()),
    value=list(SERVICES.keys())[0],
)

@pn.depends(layer_select)
def show_layer(title):
    svc = SERVICES[title]
    return (
        basemap * vap_tile(svc)
    ).opts(
        opts.WMTS(width=700, height=600, xlim=XLIM, ylim=YLIM),
        opts.Overlay(title=title),
    )

pn.Column(layer_select, show_layer)